In [ ]:
%pip install opencv-python pandas numpy matplotlib

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
img_dir = Path("Inputs/Blue Signs")

image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}

image_paths = sorted([
    path for path in img_dir.iterdir()
    if path.suffix.lower() in image_extensions
])

images = []

for image_path in image_paths:
    image = cv2.imread(str(image_path))

    if image is not None:
        images.append(image)
    else:
        print(f"Failed to load: {image_path.name}")

print("Total images loaded:", len(images))

In [ ]:
def remove_small_components(binary_mask, min_area_ratio):

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary_mask,
        connectivity=8 #include 8 directions
    )

    cleaned_mask = np.zeros_like(binary_mask)

    image_area = binary_mask.shape[0] * binary_mask.shape[1]   #shape[0]=image height , shape[1]=image width
    min_area = image_area * min_area_ratio

    for label in range(1, num_labels):  # 0 is background

        area = stats[label, cv2.CC_STAT_AREA] #get the area of connected region

        if area >= min_area:
            cleaned_mask[labels == label] = 255

    return cleaned_mask

In [ ]:
def enhance_img( image):

    image_cvt=cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    filtered = cv2.bilateralFilter(
        image_cvt,
        d=7,
        sigmaColor=40,
        sigmaSpace=40
    ) 


    filtered_cvt = cv2.cvtColor(filtered,cv2.COLOR_RGB2HSV)



    h, s, v = cv2.split(filtered_cvt)


    hue_mask = cv2.inRange(
        h,
        99,
        125
    )
    
    
    # Adaptive thresholding on Saturation
    saturation_mask = cv2.adaptiveThreshold(
        s,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        91,
        2
    )
    
    
    value_mask = cv2.inRange(
        v,
        42,
        255
    )
    
    # Combine Hue, Saturation and Value masks
    blue_mask = cv2.bitwise_and(
        hue_mask,
        saturation_mask
    )
    
    blue_mask = cv2.bitwise_and(
        blue_mask,
        value_mask
    )

    blue_mask = remove_small_components(
    blue_mask,
    min_area_ratio=0.05
    )


    # Morphological closing:
    # connects nearby white regions and fills small black gaps
    closing_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (4,4)
    )

    final_mask = cv2.morphologyEx(
        blue_mask,
        cv2.MORPH_CLOSE,
        closing_kernel,
        iterations=1
    )

   
 
    return blue_mask , filtered, final_mask




In [ ]:
def get_filled_contour_mask(binary_image):

    contours, hierarchy = cv2.findContours(
        binary_image,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    contour_mask = np.zeros_like(binary_image)


    # Select the largest contour as the traffic sign
    largest_contour = max(
        contours,
        key=cv2.contourArea
    )
    
    hull = cv2.convexHull(largest_contour)

    cv2.drawContours(
        contour_mask,
        [hull],  #cv2.drawContours() expects a collection of contours
        -1,
        255,
        thickness=cv2.FILLED  #thickness: fill the contour
    )

    return contour_mask

In [ ]:
#create empty list to store outputs
filtered=[]
threshold_img=[]
final_img=[]
contour_img=[]
segmented_img=[]

for i in range (len(images)):
    blue_img, filtered_img, final_mask=enhance_img(images[i])
   
    #contour
    contour_mask = get_filled_contour_mask(
        final_mask
    )
    
    segmented_image = cv2.bitwise_and(
        images[i],    #need two input images
        images[i],
        mask=contour_mask
    )

    threshold_img.append(blue_img)
    filtered.append(filtered_img)
    final_img.append(final_mask)
    contour_img.append(contour_mask)
    segmented_img.append(segmented_image)
    
    
    

In [ ]:
plt.figure(figsize=(21,12))
for i in range (len(filtered)):
    
    plt.subplot(4,7,i+1)
    plt.imshow(filtered[i])
    plt.title("Filtered image")
    plt.axis('off')

In [ ]:
plt.figure(figsize=(21,12))
for i in range (len(threshold_img)):
    plt.subplot(4,7,i+1)
    plt.imshow(threshold_img[i], cmap='gray')
    plt.title("Addaptive Thresholding")
    plt.axis('off')

In [ ]:
plt.figure(figsize=(21,12))
for i in range (len(final_img)):
    
    plt.subplot(4,7,i+1)
    plt.imshow(final_img[i],cmap='gray')
    plt.title("Morphological closing")
    plt.axis('off')

In [ ]:
plt.figure(figsize=(21,12))
for i in range (len(contour_img)):
    
    plt.subplot(4,7,i+1)
    plt.imshow(contour_img[i],cmap='gray')
    plt.title("Filled Contour")
    plt.axis('off')

In [ ]:
plt.figure(figsize=(21,12))
for i in range (len(segmented_img)):
    
    plt.subplot(4,7,i+1)
    plt.imshow(cv2.cvtColor(segmented_img[i], cv2.COLOR_BGR2RGB))
    plt.title("Segmented Image")
    plt.axis('off')

    plt.imsave(
    f"result img/00{i + 1}.png",
    segmented_img[i]
    )

In [ ]:
plt.figure(figsize=(20,112))
for i in range (len(filtered)):

    
    plt.subplot(28,5,i*5+1)
    plt.imshow(filtered[i])
    plt.title("Filtered image")
    plt.axis('off')
    plt.subplot(28,5,i*5+2)
    plt.imshow(threshold_img[i], cmap='gray')
    plt.title("Adaptive thresholding")
    plt.axis('off')
    plt.subplot(28,5,i*5+3)
    plt.imshow(final_img[i], cmap='gray')
    plt.title("Morphological closing")
    plt.axis('off')
    plt.subplot(28,5,i*5+4)
    plt.imshow(contour_img[i], cmap='gray')
    plt.title("Contour image")
    plt.axis('off')
    plt.subplot(28,5,i*5+5)
    plt.imshow(cv2.cvtColor(segmented_img[i], cv2.COLOR_BGR2RGB))
    plt.title("Segmented image")
    plt.axis('off')